# Matrix-Free Warm-Start ProxSVT + R4SVD for Cryo-EM

This notebook adapts warm-start proximal SVT + R4SVD algorithm to the **matrix-free**
cryo-EM setting exposed by `SketchedNormalOperator`.

We never form the dense gradient
$G(X) = A^*(A(X)-b) $. Instead, for a low-rank iterate
$X = U \operatorname{diag}(s) V^\top,$ we only use the black-box sketch products

- `op.right_matvec(U, s, V, Q)` for \(G(X)\,Q\),
- `op.left_matvec(U, s, V, S)` for \(S\,G(X)\),
- `op.both_matvecs(U, s, V, S, Q)` when both are needed.

This is enough to build matrix-free products with $Z = X - \delta\, G(X)$
namely
$ ZQ = XQ - \delta\,G(X)Q \qquad SZ = SX - \delta\,S G(X) $
which is exactly what the recycled randomized SVD step needs.

The notebook keeps the same synthetic RECOVAR setup and PPCA baseline from the demo, then runs my solver and compares subspace quality against ground truth.

In [ ]:
import os
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import jax.random as jr
import pandas as pd

warnings.filterwarnings('ignore', module='finufft')
warnings.filterwarnings('ignore', category=FutureWarning)
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

import recovar.core.fourier_transform_utils as ftu
from recovar.core import linalg
from recovar.output import metrics
from recovar.simulation import simulator
from recovar.simulation.trajectory_generation import generate_trajectory_volumes
from recovar.ppca.ppca_scale_sweep import _load_simulated_dataset, _with_trailing_separator
from recovar.ppca import ppca as ppca_mod, prior_estimation
from recovar.ppca.sketched_normal import SketchedNormalOperator

print(f'JAX devices: {jax.devices()}')

def to_real(vol_ft, vs):
    return np.asarray(ftu.get_idft3(np.asarray(vol_ft).reshape(vs)).real)

def plot_projections(vols, labels, title, axis=2, diff_ref=None, cmap='gray'):
    n = len(vols)
    fig, axes = plt.subplots(1, n, figsize=(3*n, 3))
    if n == 1:
        axes = [axes]
    for ax, vol, label in zip(axes, vols, labels):
        data = vol - diff_ref if diff_ref is not None else vol
        proj = np.sum(data, axis=axis)
        im = ax.imshow(proj, cmap=cmap)
        ax.set_title(label, fontsize=9)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    plt.show()

In [ ]:
# Core simulation / benchmark parameters
#
# Set DATASET to swap between the 'notebook' synthetic trajectory and the
# CryoBench datasets.  All datasets default to grid=64, n=20000, nl=0.1 so
# runs are comparable; override any field by editing _DATASET_PRESETS below.
DATASET = 'notebook'     # 'notebook' | 'IgG-1D' | 'IgG-RL' | 'Ribosembly' | 'Tomotwin-100'

_DEFAULT_PRESET = dict(grid=64, n_images=20000, noise_level=0.1)
_DATASET_PRESETS = {
    'notebook':     dict(_DEFAULT_PRESET),
    'IgG-1D':       dict(_DEFAULT_PRESET),
    'IgG-RL':       dict(_DEFAULT_PRESET),
    'Ribosembly':   dict(_DEFAULT_PRESET),
    'Tomotwin-100': dict(_DEFAULT_PRESET),
}
_p = _DATASET_PRESETS[DATASET]
GRID_SIZE   = _p['grid']
N_IMAGES    = _p['n_images']
NOISE_LEVEL = _p['noise_level']
N_VOLUMES   = 10          # ignored for CryoBench (uses all MRC volumes)
N_PCS       = 10
BATCH_SIZE  = 500
PPCA_ITERS  = 10
DISC_TYPE   = 'linear_interp'

# Proposed solver parameters
PROX_MAX_ITERS   = 50
PATH_LENGTH      = 10
LAMBDA_MIN_RATIO = 1e-3
STEP_SIZE        = 1.0      # tune down if the proximal iterates become unstable
PROX_TOL         = 1e-3
PROX_GRAD_TOL    = 2e-2

RSVD_EPS         = 5e-2     # adaptive stopping target for captured energy estimate
RSVD_BLOCK_SIZE  = 15
RSVD_POWER       = 1        # set to 1 for a stronger but more expensive subspace step
RSVD_MAX_RANK    = max(2 * N_PCS, 150)
NORM_PROBE_COLS  = 4
RANDOM_SEED      = 0

BASE_DIR = os.environ.get('SKETCHED_BASE_DIR', '/scratch/gpfs/GILLES/dr4514/')   # writable scratch dir; override with $SKETCHED_BASE_DIR
CRYOBENCH_VOL_ROOT = '/scratch/gpfs/GILLES/mg6942/cryobench/'  # dir containing <dataset>/vols/128_org/*.mrc
os.makedirs(BASE_DIR, exist_ok=True)


## 1. Dataset + ground truth

In [ ]:
# ---- dataset loading: 'notebook' OR CryoBench.  Both paths go through
# the same simulator.generate_synthetic_dataset call below, and every
# cached artifact (volume prefix + dataset dir) is keyed by
#   (DATASET, grid, n_images, noise_level)
# so editing any of those in cell 2 triggers automatic regeneration.
voxel_size = 4.25 * 128 / GRID_SIZE

# The cache tag — appears in every path below.
tag = f'{DATASET}_g{GRID_SIZE}_n{N_IMAGES}_nl{NOISE_LEVEL}'

if DATASET == 'notebook':
    # Trajectory volumes depend on grid only (not on n/nl).
    vol_root    = os.path.join(BASE_DIR, f'generated_volumes_g{GRID_SIZE}')
    vol_prefix  = os.path.join(vol_root, 'vol')
    ds_dir      = os.path.join(BASE_DIR, f'test_dataset_{tag}')

    if not os.path.isfile(f'{vol_prefix}0000.mrc'):
        os.makedirs(vol_root, exist_ok=True)
        print(f'Generating {N_VOLUMES} trajectory volumes at grid={GRID_SIZE} -> {vol_prefix}* ...')
        generate_trajectory_volumes(
            vol_root, grid_size=GRID_SIZE, n_volumes=N_VOLUMES,
            voxel_size=voxel_size, Bfactor=80, max_rotation_degrees=10.0,
            output_prefix=vol_prefix)
    else:
        print(f'Reusing trajectory volumes at {vol_root}/')

else:
    # CryoBench: copy source MRCs to a local dir in vol0000.mrc / vol0001.mrc layout.
    import glob, mrcfile
    from recovar.ppca.ppca_scale_sweep import find_cryobench_datasets

    vol_root    = os.path.join(BASE_DIR, 'cryobench', f'{DATASET}_vols_g{GRID_SIZE}')
    vol_prefix  = os.path.join(vol_root, 'vol')
    ds_dir      = os.path.join(BASE_DIR, 'cryobench', tag)

    if not os.path.isfile(f'{vol_prefix}0000.mrc'):
        os.makedirs(vol_root, exist_ok=True)
        datasets = find_cryobench_datasets(CRYOBENCH_VOL_ROOT)
        info = next((d for d in datasets if d['name'] == DATASET), None)
        if info is None:
            avail = [d['name'] for d in datasets]
            raise RuntimeError(
                f'Dataset {DATASET!r} not found under CRYOBENCH_VOL_ROOT={CRYOBENCH_VOL_ROOT}.\n'
                f'Available: {avail}.  Each dataset needs <root>/<name>/vols/128_org/*.mrc')
        mrc_files = sorted(glob.glob(os.path.join(info['vol_dir'], '*.mrc')))
        print(f'Preparing {len(mrc_files)} CryoBench volumes -> {vol_root}/ ...')
        for i, src in enumerate(mrc_files):
            with mrcfile.open(src, permissive=True) as m:
                data = m.data.copy()
            with mrcfile.new(f'{vol_prefix}{i:04d}.mrc', overwrite=True) as m_out:
                m_out.set_data(data)
                m_out.voxel_size = voxel_size
    else:
        print(f'Reusing CryoBench volumes at {vol_root}/')

# --- shared call: identical for both paths ---
# Cache is keyed by ds_dir which already contains (DATASET, grid, n, nl); if you
# change any of those in cell 2 the simulator re-runs automatically.
if not os.path.isfile(os.path.join(ds_dir, f'particles.{GRID_SIZE}.mrcs')):
    print(f'Simulating {N_IMAGES} images, noise={NOISE_LEVEL} -> {ds_dir}/ ...')
    np.random.seed(42)
    simulator.generate_synthetic_dataset(
        ds_dir, voxel_size, vol_prefix, N_IMAGES,
        grid_size=GRID_SIZE, noise_level=NOISE_LEVEL, noise_model='radial1',
        contrast_std=0.0, noise_scale_std=0.0,
        dataset_params_option='uniform', disc_type=DISC_TYPE,
        trailing_zero_format_in_vol_name=True,
        put_extra_particles=False, percent_outliers=0.0)
else:
    print(f'Reusing dataset at {ds_dir}')

cryo, sim_info, gt, noise_var = _load_simulated_dataset(
    _with_trailing_separator(ds_dir), GRID_SIZE, N_IMAGES, lazy=False)
vs = cryo.volume_shape
vol_size = int(np.prod(vs))
n_images = int(cryo.n_images)
print(f'[{tag}] grid={cryo.grid_size}, n_images={n_images}, vol_size={vol_size}')


In [ ]:
# Convert GT states to real space
state_vols = [to_real(gt.volumes[i], vs) for i in range(gt.volumes.shape[0])]

# Show a selection of states — projections along z and x
n_show = min(5, len(state_vols))
step = max(1, len(state_vols) // n_show)
idx = list(range(0, len(state_vols), step))[:n_show]

plot_projections([state_vols[i] for i in idx],
                 [f'State {i}' for i in idx],
                 f'GT states — projection along z  ({N_VOLUMES} total)', axis=2)

plot_projections([state_vols[i] for i in idx],
                 [f'State {i}' for i in idx],
                 f'GT states — projection along x', axis=0)

# Difference from state 0 — makes the heterogeneity visible
plot_projections([state_vols[i] for i in idx],
                 [f'{i} − 0' for i in idx],
                 'Difference from state 0 — projection along z',
                 axis=2, diff_ref=state_vols[0], cmap='RdBu_r')

plot_projections([state_vols[i] for i in idx],
                 [f'{i} − 0' for i in idx],
                 'Difference from state 0 — projection along x',
                 axis=0, diff_ref=state_vols[0], cmap='RdBu_r')

In [ ]:
# GT eigenvectors (Fourier) + true per-image coordinates #
# Ground-truth SVD data from the synthetic generator
U_gt_all, s_gt_all, Vh_gt = gt.get_vol_svd()
gt_mean = gt.get_mean()   # Fourier domain
probs = gt.get_probs_of_state()

# True V: gt.get_vol_svd() is SVD of sqrt(p) * (vol - mean),
# so vol[k] - mean = U @ diag(s) @ Vh[:,k] / sqrt(p[k])
assign = np.array(sim_info['image_assignment'])
V_true = (np.asarray(Vh_gt[:N_PCS, :]) / np.sqrt(probs)[None, :]).T[assign].real.astype(np.float32)

# Convert GT eigenvectors to real space for the matrix-free API
vol_norm = np.sqrt(vol_size)
U_gt_real = np.asarray(
    ftu.get_idft3(U_gt_all[:, :N_PCS].T.reshape(N_PCS, *vs))
).real.reshape(N_PCS, vol_size).T * vol_norm

print(f'GT: {N_PCS} PCs, top eigenvalues: {s_gt_all[:5]}')

## 2. Matrix-free Warm-Start ProxSVT + R4SVD

The dense version of my old experiments used explicit products with
$ Z = X - \delta G(X)$. Here we only need the sketched products

$$ ZQ = XQ - \delta\,G(X)Q \qquad SZ = SX - \delta\,S G(X) $$

which are available from `SketchedNormalOperator` once \(X\) is stored in low-rank form.

A small implementation note: because $\|Z\|_F $ is unavailable without forming $Z$, the R4SVD stopping test uses a **Hutchinson-style estimate**

$$ \|Z\|_F^2 \approx \frac{1}{t}\|Z\Omega\|_F^2 $$

with a small Gaussian probe matrix $\Omega$.
This gives an adaptive captured-energy proxy for the recycled range expansion.

In [ ]:
LEFT_SCALE = vol_size / 2.0
def _empty_factors(vol_size, n_images):
    return (
        np.zeros((vol_size, 0), dtype=np.float32),
        np.zeros((0,), dtype=np.float32),
        np.zeros((n_images, 0), dtype=np.float32),
    )

def _fro_sq(A):
    A = np.asarray(A, dtype=np.float32)
    return float(np.sum(A * A))

def _orth(X):
    X = np.asarray(X, dtype=np.float32)
    if X.size == 0:
        return np.zeros((X.shape[0], 0), dtype=np.float32)
    Q, _ = np.linalg.qr(X, mode='reduced')
    return np.asarray(Q, dtype=np.float32)

def low_rank_right(U, s, V, Q, vol_size):
    Q = np.asarray(Q, dtype=np.float32)
    if U is None or U.size == 0 or len(s) == 0:
        return np.zeros((vol_size, Q.shape[1]), dtype=np.float32)
    return (U * s[None, :]) @ (V.T @ Q)

def low_rank_left(U, s, V, S, n_images):
    S = np.asarray(S, dtype=np.float32)
    if U is None or U.size == 0 or len(s) == 0:
        return np.zeros((S.shape[0], n_images), dtype=np.float32)
    return (S @ (U * s[None, :])) @ V.T

def left_matvec_scaled(op, U, s, V, S, left_scale=LEFT_SCALE):
    return np.array(op.left_matvec(U, s, V, S), dtype=np.float32, copy=True) / left_scale

def right_matvec_scaled(op, U, s, V, Q):
    return np.array(op.right_matvec(U, s, V, Q), dtype=np.float32, copy=True)

def both_matvecs_scaled(op, U, s, V, S, Q, left_scale=LEFT_SCALE):
    SG, GQ = op.both_matvecs(U, s, V, S, Q)
    SG = np.array(SG, dtype=np.float32, copy=True) / left_scale
    GQ = np.array(GQ, dtype=np.float32, copy=True)
    return SG, GQ

def apply_Z_right(op, U, s, V, delta, Q, vol_size):
    if len(s) == 0:
        XQ = np.zeros((vol_size, Q.shape[1]), dtype=np.float32)
    else:
        XQ = (U * s) @ (V.T @ Q)
    GQ = right_matvec_scaled(op, U, s, V, Q)
    return XQ - delta * GQ

def apply_Z_left(op, U, s, V, delta, S, n_images):
    if len(s) == 0:
        SX = np.zeros((S.shape[0], n_images), dtype=np.float32)
    else:
        SX = (S @ (U * s)) @ V.T
    SG = left_matvec_scaled(op, U, s, V, S)
    return SX - delta * SG

def both_Z_matvecs(op, U, s, V, delta, S, Q, vol_size, n_images):
    SG, GQ = both_matvecs_scaled(op, U, s, V, S, Q)
    left = low_rank_left(U, s, V, S, n_images) - delta * SG
    right = low_rank_right(U, s, V, Q, vol_size) - delta * GQ
    return left, right

def low_rank_inner(U1, s1, V1, U2, s2, V2):
    if len(s1) == 0 or len(s2) == 0:
        return 0.0
    UU = U1.T @ U2
    VV = V1.T @ V2
    return float(np.sum(UU * VV * (s1[:, None] * s2[None, :])))

def low_rank_relchg(U_old, s_old, V_old, U_new, s_new, V_new):
    old_sq = float(np.sum(np.asarray(s_old, dtype=np.float32) ** 2))
    new_sq = float(np.sum(np.asarray(s_new, dtype=np.float32) ** 2))
    diff_sq = max(old_sq + new_sq - 2.0 * low_rank_inner(U_old, s_old, V_old, U_new, s_new, V_new), 0.0)
    return np.sqrt(diff_sq) / max(np.sqrt(old_sq), 1e-12)

def estimate_fro_norm_sq_Z(op, U, s, V, delta, n_images, vol_size, rng, probe_cols=4):
    Omega = rng.normal(size=(n_images, probe_cols)).astype(np.float32)
    Y = apply_Z_right(op, U, s, V, delta, Omega, vol_size)
    return _fro_sq(Y) / probe_cols

def expand_block(op, U, s, V, delta, n_images, vol_size, basis, rng, block_size=8, n_power=0):
    Omega = rng.normal(size=(n_images, block_size)).astype(np.float32)
    Y = apply_Z_right(op, U, s, V, delta, Omega, vol_size)

    if basis.size:
        Y = Y - basis @ (basis.T @ Y)

    for _ in range(n_power):
        T = _orth(Y)
        W = apply_Z_left(op, U, s, V, delta, T.T, n_images).T
        Y = apply_Z_right(op, U, s, V, delta, W, vol_size)
        if basis.size:
            Y = Y - basis @ (basis.T @ Y)

    Q0 = _orth(Y)
    if basis.size and Q0.size:
        Q0 = _orth(Q0 - basis @ (basis.T @ Q0))
    return Q0

def proximal_r4svd_step_matrix_free(
    op,
    U,
    s,
    V,
    lam,
    delta,
    n_images,
    vol_size,
    rng,
    block_size=10,
    n_power=1,
    max_rank=60,
    oversamp_guard=5,
    stop_margin=0.10,
    stable_rounds_needed=2,
):
    if U is None:
        U, s, V = _empty_factors(vol_size, n_images)

    basis = U.copy()
    if basis.size:
        B = apply_Z_left(op, U, s, V, delta, basis.T, n_images)
    else:
        B = np.zeros((0, n_images), dtype=np.float32)

    stable_rounds = 0
    last_keep = -1
    sb = np.zeros((0,), dtype=np.float32)

    while basis.shape[1] < max_rank:
        blk = min(block_size, max_rank - basis.shape[1])
        Q0 = expand_block(
            op, U, s, V, delta, n_images, vol_size, basis, rng,
            block_size=blk, n_power=n_power
        )
        if Q0.size == 0:
            break

        B0 = apply_Z_left(op, U, s, V, delta, Q0.T, n_images)
        basis = np.concatenate([basis, Q0], axis=1)
        B = np.vstack([B, B0])

        Ub, sb, Vhb = np.linalg.svd(B, full_matrices=False)
        thresh = lam * delta
        keep = int(np.sum(sb > thresh))

        # stop when we have a few extra singular values below threshold
        if keep + oversamp_guard < len(sb):
            tail_sv = sb[keep + oversamp_guard - 1]
            if tail_sv <= (1.0 + stop_margin) * thresh and keep == last_keep:
                stable_rounds += 1
            else:
                stable_rounds = 0
            if stable_rounds >= stable_rounds_needed:
                break

        last_keep = keep

        if Q0.shape[1] < blk:
            break

    if basis.shape[1] == 0:
        U_new, s_new, V_new = _empty_factors(vol_size, n_images)
        info = {
            "rank_before": int(len(s)),
            "rank_after": 0,
            "basis_rank": 0,
            "sb": np.zeros((0,), dtype=np.float32),
            "stop_reason": "empty_basis",
        }
        return U_new, s_new, V_new, info

    Ub, sb, Vhb = np.linalg.svd(B, full_matrices=False)
    keep_mask = sb > lam * delta

    if not np.any(keep_mask):
        U_new, s_new, V_new = _empty_factors(vol_size, n_images)
        info = {
            "rank_before": int(len(s)),
            "rank_after": 0,
            "basis_rank": int(basis.shape[1]),
            "sb": sb.copy(),
            "stop_reason": "thresholded_to_zero",
        }
        return U_new, s_new, V_new, info

    U_new = (basis @ Ub[:, keep_mask]).astype(np.float32)
    s_new = (sb[keep_mask] - lam * delta).astype(np.float32)
    V_new = Vhb[keep_mask, :].T.astype(np.float32)

    info = {
        "rank_before": int(len(s)),
        "rank_after": int(len(s_new)),
        "basis_rank": int(basis.shape[1]),
        "sb": sb.copy(),
        "stop_reason": "threshold_stable_or_rank_cap",
    }
    return U_new, s_new, V_new, info

def estimate_lambda_max(op, n_images, vol_size, rng, n_iter=8):
    U0, s0, V0 = _empty_factors(vol_size, n_images)
    q = rng.normal(size=(n_images, 1)).astype(np.float32)
    q = q / max(np.linalg.norm(q), 1e-12)

    sigma = 0.0
    for _ in range(n_iter):
        y = right_matvec_scaled(op, U0, s0, V0, q)
        y_norm = np.linalg.norm(y)
        if y_norm == 0.0:
            return 0.0
        y = y / y_norm

        q = left_matvec_scaled(op, U0, s0, V0, y.T).T
        sigma = np.linalg.norm(q)
        if sigma == 0.0:
            return 0.0
        q = q / sigma

    return float(sigma)

def real_pcs_to_fourier(U_real, vs):
    U_real = np.asarray(U_real)
    vol_size = int(np.prod(vs))

    if U_real.ndim != 2:
        raise ValueError(f"U_real must be 2D, got shape {U_real.shape}")

    if U_real.shape[0] != vol_size:
        raise ValueError(
            f"U_real has shape {U_real.shape}, but expected first dimension = prod(vs) = {vol_size}"
        )

    r = U_real.shape[1]
    if r == 0:
        return np.zeros((vol_size, 0), dtype=np.complex64)

    U_real_flat = jnp.asarray(U_real, dtype=jnp.float32)   # shape: (vol_size, r)
    U_fourier = np.asarray(linalg.batch_dft3(U_real_flat, vs, r))

    return U_fourier / np.sqrt(vol_size)

def prox_svt_r4svd_single_lambda_cold_start(
    op,
    lam,
    vol_size,
    n_images,
    step=1.0,
    max_iters=25,
    tol=1e-3,
    grad_tol=2e-2,
    block_size=15,
    n_power=1,
    max_rank=30,
    norm_probe_cols=4,
    rng_seed=0,
    oversamp_guard=5,
    stop_margin=0.10,
    stable_rounds_needed=2,
):
    """
    Solve one fixed-lambda problem from a cold start (U=s=V=0).

    Returns:
        {
            "lambda": float,
            "solution": (U, s, V),
            "history": [dict, ...],
        }
    """
    rng = np.random.default_rng(rng_seed)
    lam = float(lam)

    # Cold start
    U, s, V = _empty_factors(vol_size, n_images)

    # Stopping proxy baseline at zero iterate
    Q_stop = rng.normal(size=(n_images, norm_probe_cols)).astype(np.float32)
    G0Q = np.array(op.right_matvec(U, s, V, Q_stop), dtype=np.float32, copy=True)
    grad0_proxy = np.sqrt(_fro_sq(G0Q) / norm_probe_cols) + 1e-20

    hist = []
    t0 = time.time()

    for it in range(max_iters):
        U_new, s_new, V_new, info = proximal_r4svd_step_matrix_free(
            op,
            U,
            s,
            V,
            lam,
            step,
            n_images,
            vol_size,
            rng,
            block_size=block_size,
            n_power=n_power,
            max_rank=max_rank,
            oversamp_guard=oversamp_guard,
            stop_margin=stop_margin,
            stable_rounds_needed=stable_rounds_needed,
        )

        relchg = low_rank_relchg(U, s, V, U_new, s_new, V_new)

        GQ = np.array(op.right_matvec(U_new, s_new, V_new, Q_stop), dtype=np.float32, copy=True)
        grad_proxy = np.sqrt(_fro_sq(GQ) / norm_probe_cols) / grad0_proxy

        hist.append({
            "iter": it + 1,
            "lambda": lam,
            "rank": int(len(s_new)),
            "basis_rank": int(info["basis_rank"]),
            "top_sv": float(info["sb"][0]) if len(info["sb"]) else 0.0,
            "sv2": float(info["sb"][1]) if len(info["sb"]) > 1 else 0.0,
            "sv5": float(info["sb"][4]) if len(info["sb"]) > 4 else 0.0,
            "sv10": float(info["sb"][9]) if len(info["sb"]) > 9 else 0.0,
            "n_above_thresh": int(np.sum(info["sb"] > lam * step)),
            "relchg": float(relchg),
            "grad_proxy": float(grad_proxy),
            "stop_reason": info["stop_reason"],
            "elapsed_sec": float(time.time() - t0),
        })

        U, s, V = U_new, s_new, V_new

        if relchg < tol or grad_proxy < grad_tol:
            break

    return {
        "lambda": lam,
        "solution": (U.copy(), s.copy(), V.copy()),
        "history": hist,
    }

## 3. Construct the sketched normal operator and run PPCA
**Relative variance** measures how well estimated PCs span the GT heterogeneity subspace.

Let $U_{GT}$ and $s_{GT}$ be the GT eigenvectors and eigenvalues from the SVD of
the probability-weighted centered volume matrix.  For estimated PCs $\hat{U}$,
the captured variance up to $k$ PCs is

$$\text{CaptVar}(k) = \sum_{j=1}^k \|\hat{U}_j^H\, U_{GT}\, \text{diag}(\sqrt{s_{GT}})\|^2,$$

and the **relative variance** is $\text{RelVar}(k) = \text{CaptVar}(k)\, /\, \sum_j s_{GT,j}$.

RelVar = 1.0 means the estimated subspace captures all GT variance.
A higher RelVar at fewer PCs means better quality.

In [ ]:
# Operator without the radial Fourier prior (stock sketched normal op).
op = SketchedNormalOperator(cryo, gt_mean, batch_size=BATCH_SIZE, disc_type=DISC_TYPE)

W_init = jr.normal(jr.PRNGKey(0), (vol_size, N_PCS), dtype=jnp.float32)
W_init = linalg.batch_dft3(W_init, vs, N_PCS)
W_prior = prior_estimation.make_gt_prior_from_variance_total(
    gt.get_fourier_variances(contrasted=False),
    N_PCS,
    vs,
)['W_prior']

t0 = time.time()
U_ppca, S_ppca, W_ppca, _, _ = ppca_mod.EM(
    cryo, gt_mean, W_init, W_prior,
    EM_iter=PPCA_ITERS, U_gt=U_gt_all, S_gt=s_gt_all,
    disc_type_mean=DISC_TYPE, disc_type=DISC_TYPE, contrast_mode='none')
dt_ppca = time.time() - t0

rv_ppca = metrics.captured_variance(U_ppca, U_gt_all, s_gt_all)
relvar_ppca = np.asarray(metrics.relative_variance_from_captured_variance(rv_ppca, s_gt_all))
print(f'PPCA EM: {PPCA_ITERS} iters in {dt_ppca:.1f}s, relvar@{N_PCS} = {relvar_ppca[-1]:.4f}')


## 4. Run the proposed matrix-free warm-start ProxSVT + R4SVD path

In [ ]:
# SVT threshold for the no-prior run.  Tuned so the solver activates near
# rank 10 on the notebook benchmark; the equivalent knob for the prior run
# is lam_test_prior in cell 16 (the prior changes the operator scale, so the
# SVT threshold there has to be larger).
lam_test = 4.11e-4

t0 = time.time()
single_out = prox_svt_r4svd_single_lambda_cold_start(
    op,
    lam=lam_test,
    vol_size=vol_size,
    n_images=n_images,
    step=STEP_SIZE,
    max_iters=PROX_MAX_ITERS,
    tol=PROX_TOL,
    grad_tol=PROX_GRAD_TOL,
    block_size=RSVD_BLOCK_SIZE,
    n_power=RSVD_POWER,
    max_rank=RSVD_MAX_RANK,
    norm_probe_cols=NORM_PROBE_COLS,
    rng_seed=RANDOM_SEED,
    oversamp_guard=5,
    stop_margin=0.10,
    stable_rounds_needed=2,
)
dt_single = time.time() - t0

print(f"lambda = {single_out['lambda']:.4e}")
print(f"total time = {dt_single:.1f}s")


In [ ]:
# Evaluate the no-prior solution against GT up front so later cells can use
# score_single directly without re-running anything.
U_est_real, s_est, V_est = single_out["solution"]
U_est_fourier = real_pcs_to_fourier(U_est_real, vs)
rv_single     = metrics.captured_variance(U_est_fourier, U_gt_all, s_gt_all)
relvar_single = np.asarray(metrics.relative_variance_from_captured_variance(rv_single, s_gt_all))
score_single  = relvar_single[min(N_PCS, len(relvar_single)) - 1] if len(relvar_single) > 0 else 0.0

print(f"[no-prior] lambda={single_out['lambda']:.4e}  rank={len(s_est)}  rv@{N_PCS}={score_single:.4f}")


## 5. Add a radial Fourier prior and re-run

Section 4 runs the solver with only the nuclear-norm penalty.  Adding a
**radial Fourier prior** — the same machinery PPCA uses internally — lets
the sketched solver match or beat PPCA on this benchmark.

The prior is
$ R(X) = \tfrac{\lambda_{\text{prior}}}{2}\,\|D X\|_F^2$
with $D$ Fourier-diagonal and $D^2(|k|) = 1 / (\text{shell-power}(|k|) + \varepsilon)$
built from the ground-truth Fourier variance.  Its gradient is added to the
normal residual directly inside `SketchedNormalOperator` (via the new
`D2_fourier` / `prior_lambda` constructor args) — the proximal solver itself
is unchanged.


In [ ]:
from recovar.reconstruction import regularization
from recovar import utils

# ---- radial D^2 from GT Fourier variance ----
_fv    = np.asarray(gt.get_fourier_variances(contrasted=False))  # (V,) real
_shells = np.asarray(regularization.batch_average_over_shells(
    jnp.asarray(_fv[None, :]), vs, 0))[0]
_eps   = 1e-4 * float(_shells.max())
_D2_r  = 1.0 / (_shells + _eps)
D2_fourier = np.asarray(utils.batch_make_radial_image(
    jnp.asarray(_D2_r[None, :]), vs, True))[0].reshape(-1).astype(np.float32)

PRIOR_LAMBDA = 1.0   # strength of the radial prior; 1.0 is a sensible default

# Prior now lives inside SketchedNormalOperator — no wrapper class needed.
op_prior = SketchedNormalOperator(
    cryo, gt_mean, batch_size=BATCH_SIZE, disc_type=DISC_TYPE,
    D2_fourier=D2_fourier, prior_lambda=PRIOR_LAMBDA,
)
print(f'[prior] lam={PRIOR_LAMBDA}  D2_fourier range [{D2_fourier.min():.3e}, {D2_fourier.max():.3e}]')


In [ ]:
# Re-run the exact same solver on the prior-augmented operator.
# The operator magnitude changes when D^2 X is added, so the SVT threshold
# needs its own tuning.  lam_test_prior is the threshold for the prior run;
# lam_test (cell 12) was for the no-prior run.  Both just threshold
# singular values — the value that triggers ~rank-10 output differs because
# the operator norms differ.
lam_test_prior = 1.0

t0 = time.time()
prior_out = prox_svt_r4svd_single_lambda_cold_start(
    op_prior,
    lam=lam_test_prior,
    vol_size=vol_size,
    n_images=n_images,
    step=STEP_SIZE,
    max_iters=PROX_MAX_ITERS,
    tol=PROX_TOL,
    grad_tol=PROX_GRAD_TOL,
    block_size=RSVD_BLOCK_SIZE,
    n_power=RSVD_POWER,
    max_rank=RSVD_MAX_RANK,
    norm_probe_cols=NORM_PROBE_COLS,
    rng_seed=RANDOM_SEED,
)
dt_prior = time.time() - t0

U_p, s_p, V_p = prior_out['solution']
U_p_f = real_pcs_to_fourier(U_p, vs)
rv_prior      = metrics.captured_variance(U_p_f, U_gt_all, s_gt_all)
relvar_prior  = np.asarray(metrics.relative_variance_from_captured_variance(rv_prior, s_gt_all))
score_prior   = relvar_prior[min(N_PCS, len(relvar_prior)) - 1] if len(relvar_prior) > 0 else 0.0

print(f'[prior-aug] rank={len(s_p)}  rv@{N_PCS}={score_prior:.4f}  (time {dt_prior:.1f}s)')
print(f'[baseline ] PPCA-EM rv@{N_PCS}={relvar_ppca[-1]:.4f}')
print(f'[no-prior ] ProxSVT+R4SVD rv@{N_PCS}={score_single:.4f}')


In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

single_hist_df = pd.DataFrame(single_out["history"])
print(single_hist_df)

In [ ]:
# No-prior cold-start summary + plot.
print(f"lambda = {single_out['lambda']:.4e}")
print(f"rank = {len(s_est)}")
print(f"relvar@{N_PCS} = {score_single:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
if len(relvar_single) > 0:
    ax.plot(
        range(1, len(relvar_single) + 1),
        relvar_single,
        'o-',
        label=f'ProxSVT+R4SVD (λ={single_out["lambda"]:.2e})',
        linewidth=2.5,
        markersize=4,
    )
ax.set_xlabel('Number of PCs')
ax.set_ylabel('Cumulative relative variance')
ax.set_title('Cold-start subspace quality vs ground truth')
ax.set_ylim(0, 0.2)
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Final comparison plot: PPCA baseline vs. sketched solver with / without the radial prior.
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.plot(range(1, len(relvar_ppca) + 1), relvar_ppca, 'o-',
        label=f'PPCA EM ({PPCA_ITERS} iters, {dt_ppca:.0f}s), rv@{N_PCS}={relvar_ppca[-1]:.3f}',
        linewidth=2, markersize=4)

if len(relvar_single) > 0:
    ax.plot(range(1, len(relvar_single) + 1), relvar_single, 's-',
            label=f'ProxSVT+R4SVD no prior (λ={single_out["lambda"]:.2e}), rv@{N_PCS}={score_single:.3f}',
            linewidth=2, markersize=4)

if 'relvar_prior' in globals() and len(relvar_prior) > 0:
    ax.plot(range(1, len(relvar_prior) + 1), relvar_prior, '^-',
            label=f'ProxSVT+R4SVD + radial_D prior (λ_prior={PRIOR_LAMBDA}), rv@{N_PCS}={score_prior:.3f}',
            linewidth=2.5, markersize=4)

ax.set_xlabel('Number of PCs')
ax.set_ylabel('Cumulative relative variance')
ax.set_title(f'[{DATASET}] Subspace quality vs ground truth')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()
